# Medical RAG - Benchmark Analysis

This notebook analyzes the performance of different search strategies across multiple benchmark runs.

## Metrics Analyzed
- **Latency**: Response time (avg, median, p95, p99)
- **Precision**: Proportion of retrieved documents that are relevant
- **Recall**: Proportion of relevant documents that are retrieved
- **F1 Score**: Harmonic mean of precision and recall

## Search Strategies
1. **IVFFlat Only**: Pure vector similarity (baseline)
2. **IVFFlat + NER Boost**: Vector + entity awareness
3. **IVFFlat + tsvector**: Vector + keyword search
4. **IVFFlat + NER + tsvector**: Full hybrid (all signals)

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Set figure size defaults
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

## Load Benchmark Data

In [ ]:
# File paths
benchmark_files = {
    '30 queries': 'recall analysis/benchmark_results_30queries_recall_metrics.json',
    '50 queries': 'recall analysis/benchmark_50queries_recall_metrics.json',
    '100 queries': 'recall analysis/benchmark_100queries_recall_metrics.json'
}

# Load all benchmark files
benchmarks = {}
for name, filepath in benchmark_files.items():
    try:
        with open(filepath, 'r') as f:
            benchmarks[name] = json.load(f)
        print(f"✓ Loaded: {name}")
    except FileNotFoundError:
        print(f"✗ Not found: {filepath}")

print(f"\nTotal files loaded: {len(benchmarks)}")

## Extract Latency Metrics

In [ ]:
def extract_latency_metrics(benchmarks):
    """
    Extract latency metrics from benchmark results.
    """
    rows = []
    
    for benchmark_name, data in benchmarks.items():
        strategy_metrics = data.get('strategy_metrics', {})
        
        for strategy_key, metrics in strategy_metrics.items():
            row = {
                'Benchmark': benchmark_name,
                'Strategy': metrics.get('strategy', strategy_key),
                'Queries': metrics.get('num_queries', 0),
                'Avg Latency (ms)': round(metrics.get('avg_latency_ms', 0), 2),
                'Median Latency (ms)': round(metrics.get('median_latency_ms', 0), 2),
                'P95 Latency (ms)': round(metrics.get('p95_latency_ms', 0), 2),
                'P99 Latency (ms)': round(metrics.get('p99_latency_ms', 0), 2),
                'Std Dev (ms)': round(metrics.get('std_latency_ms', 0), 2),
                'Min Latency (ms)': round(metrics.get('min_latency_ms', 0), 2),
                'Max Latency (ms)': round(metrics.get('max_latency_ms', 0), 2)
            }
            rows.append(row)
    
    return pd.DataFrame(rows)

latency_df = extract_latency_metrics(benchmarks)
print("\n📊 Latency Metrics Summary:")
latency_df

## Extract Precision & Recall Metrics

In [ ]:
def extract_precision_recall_metrics(benchmarks):
    """
    Extract precision, recall, and F1 metrics from benchmark results.
    """
    rows = []
    
    strategy_map = {
        'ivfflat_only': 'IVFFlat Only',
        'ivfflat_ner_boost': 'IVFFlat + NER Boost',
        'ivfflat_tsvector': 'IVFFlat + tsvector',
        'ivfflat_ner_tsvector': 'IVFFlat + NER + tsvector'
    }
    
    for benchmark_name, data in benchmarks.items():
        recall_metrics = data.get('recall_metrics', {})
        
        for strategy_key, queries in recall_metrics.items():
            if not isinstance(queries, list):
                continue
            
            # Calculate average metrics across all queries
            precisions = [q.get('precision', 0) for q in queries if isinstance(q, dict)]
            recalls = [q.get('recall', 0) for q in queries if isinstance(q, dict)]
            f1s = [q.get('f1', 0) for q in queries if isinstance(q, dict)]
            
            if precisions:  # Only add if we have data
                row = {
                    'Benchmark': benchmark_name,
                    'Strategy': strategy_map.get(strategy_key, strategy_key),
                    'Queries': len(queries),
                    'Avg Precision': round(np.mean(precisions), 4),
                    'Avg Recall': round(np.mean(recalls), 4),
                    'Avg F1': round(np.mean(f1s), 4),
                    'Min Precision': round(np.min(precisions), 4),
                    'Max Precision': round(np.max(precisions), 4),
                    'Min Recall': round(np.min(recalls), 4),
                    'Max Recall': round(np.max(recalls), 4)
                }
                rows.append(row)
    
    return pd.DataFrame(rows)

precision_recall_df = extract_precision_recall_metrics(benchmarks)
print("\n📊 Precision & Recall Metrics Summary:")
precision_recall_df

## Presentation-Ready Tables

### Table 1: Latency Comparison Across Query Counts

In [ ]:
# Pivot table for latency - Average Latency
latency_pivot = latency_df.pivot_table(
    index='Strategy',
    columns='Benchmark',
    values='Avg Latency (ms)',
    aggfunc='first'
)

print("\n=" * 80)
print("TABLE 1: Average Latency (ms) by Strategy and Query Count")
print("=" * 80)
print(latency_pivot.to_string())
print("\n")

# Style for presentation
latency_pivot_styled = latency_pivot.style.background_gradient(
    cmap='RdYlGn_r', axis=None
).format("{:.2f}")

latency_pivot_styled

### Table 2: Median Latency Comparison

In [ ]:
# Pivot table for median latency
median_latency_pivot = latency_df.pivot_table(
    index='Strategy',
    columns='Benchmark',
    values='Median Latency (ms)',
    aggfunc='first'
)

print("\n=" * 80)
print("TABLE 2: Median Latency (ms) by Strategy and Query Count")
print("=" * 80)
print(median_latency_pivot.to_string())
print("\n")

median_latency_pivot.style.background_gradient(
    cmap='RdYlGn_r', axis=None
).format("{:.2f}")

### Table 3: P95 Latency (95th Percentile)

In [ ]:
# Pivot table for P95 latency
p95_latency_pivot = latency_df.pivot_table(
    index='Strategy',
    columns='Benchmark',
    values='P95 Latency (ms)',
    aggfunc='first'
)

print("\n=" * 80)
print("TABLE 3: P95 Latency (ms) by Strategy and Query Count")
print("=" * 80)
print(p95_latency_pivot.to_string())
print("\n")

p95_latency_pivot.style.background_gradient(
    cmap='RdYlGn_r', axis=None
).format("{:.2f}")

### Table 4: Precision Comparison

In [ ]:
# Pivot table for precision
precision_pivot = precision_recall_df.pivot_table(
    index='Strategy',
    columns='Benchmark',
    values='Avg Precision',
    aggfunc='first'
)

print("\n=" * 80)
print("TABLE 4: Average Precision by Strategy and Query Count")
print("=" * 80)
print(precision_pivot.to_string())
print("\n")

precision_pivot.style.background_gradient(
    cmap='RdYlGn', axis=None
).format("{:.4f}")

### Table 5: Recall Comparison

In [ ]:
# Pivot table for recall
recall_pivot = precision_recall_df.pivot_table(
    index='Strategy',
    columns='Benchmark',
    values='Avg Recall',
    aggfunc='first'
)

print("\n=" * 80)
print("TABLE 5: Average Recall by Strategy and Query Count")
print("=" * 80)
print(recall_pivot.to_string())
print("\n")

recall_pivot.style.background_gradient(
    cmap='RdYlGn', axis=None
).format("{:.4f}")

### Table 6: F1 Score Comparison

In [ ]:
# Pivot table for F1
f1_pivot = precision_recall_df.pivot_table(
    index='Strategy',
    columns='Benchmark',
    values='Avg F1',
    aggfunc='first'
)

print("\n=" * 80)
print("TABLE 6: Average F1 Score by Strategy and Query Count")
print("=" * 80)
print(f1_pivot.to_string())
print("\n")

f1_pivot.style.background_gradient(
    cmap='RdYlGn', axis=None
).format("{:.4f}")

## Comprehensive Summary Table

In [ ]:
if not ivfflat_df.empty:
    print("=" * 90)
    print("IVFFlat OPTIMIZATION RESULTS")
    print("=" * 90)
    print()
    
    # 1. Fastest configuration
    fastest = ivfflat_df.loc[ivfflat_df['avg_query_time_ms'].idxmin()]
    print("🏆 FASTEST (Lowest Latency):")
    print(f"   nlist={int(fastest['nlist'])}, nprobe={int(fastest['nprobe'])}")
    print(f"   Latency: {fastest['avg_query_time_ms']:.2f}ms")
    print(f"   Recall: {fastest['avg_recall']:.3f}")
    print(f"   Speedup vs exact: {ivfflat_data['exact_search_avg_ms'] / fastest['avg_query_time_ms']:.1f}x")
    print()
    
    # 2. Best recall configuration
    best_recall = ivfflat_df.loc[ivfflat_df['avg_recall'].idxmax()]
    print("🎯 BEST RECALL (Highest Quality):")
    print(f"   nlist={int(best_recall['nlist'])}, nprobe={int(best_recall['nprobe'])}")
    print(f"   Latency: {best_recall['avg_query_time_ms']:.2f}ms")
    print(f"   Recall: {best_recall['avg_recall']:.3f}")
    if best_recall['avg_recall'] >= 1.0:
        print(f"   ✨ Perfect recall! Matches exact search.")
    print()
    
    # 3. Best balanced (high recall with reasonable latency)
    # Filter for recall >= 0.95 and find fastest among them
    high_recall = ivfflat_df[ivfflat_df['avg_recall'] >= 0.95]
    if not high_recall.empty:
        balanced = high_recall.loc[high_recall['avg_query_time_ms'].idxmin()]
        print("⚖️  RECOMMENDED BALANCE (Recall ≥ 0.95, Lowest Latency):")
        print(f"   nlist={int(balanced['nlist'])}, nprobe={int(balanced['nprobe'])}")
        print(f"   Latency: {balanced['avg_query_time_ms']:.2f}ms")
        print(f"   Recall: {balanced['avg_recall']:.3f}")
        print(f"   Speedup vs exact: {ivfflat_data['exact_search_avg_ms'] / balanced['avg_query_time_ms']:.1f}x")
        print(f"   💡 Good tradeoff: {balanced['avg_recall']*100:.1f}% recall at {balanced['avg_query_time_ms']:.2f}ms")
        print()
    
    # 4. Efficiency analysis
    # Find best speedup while maintaining recall >= 0.90
    efficient = ivfflat_df[ivfflat_df['avg_recall'] >= 0.90].copy()
    if not efficient.empty:
        efficient['speedup'] = ivfflat_data['exact_search_avg_ms'] / efficient['avg_query_time_ms']
        best_efficient = efficient.loc[efficient['speedup'].idxmax()]
        print("⚡ MOST EFFICIENT (Best Speedup with Recall ≥ 0.90):")
        print(f"   nlist={int(best_efficient['nlist'])}, nprobe={int(best_efficient['nprobe'])}")
        print(f"   Latency: {best_efficient['avg_query_time_ms']:.2f}ms")
        print(f"   Recall: {best_efficient['avg_recall']:.3f}")
        print(f"   Speedup: {best_efficient['speedup']:.1f}x faster than exact search")
        print()
    
    print("=" * 90)
    print("RECOMMENDATIONS:")
    print("=" * 90)
    if not high_recall.empty:
        print(f"✅ For production: nlist={int(balanced['nlist'])}, nprobe={int(balanced['nprobe'])}")
        print(f"   → {balanced['avg_recall']*100:.1f}% recall, {balanced['avg_query_time_ms']:.2f}ms latency")
    print(f"⚡ For maximum speed: nlist={int(fastest['nlist'])}, nprobe={int(fastest['nprobe'])}")
    print(f"   → {fastest['avg_recall']*100:.1f}% recall, {fastest['avg_query_time_ms']:.2f}ms latency")
    print("=" * 90)
else:
    print("No IVFFlat data available for optimization")

## Visualizations

### Figure 1: Average Latency Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Prepare data for grouped bar chart
benchmarks_order = ['30 queries', '50 queries', '100 queries']
strategies = latency_df['Strategy'].unique()

x = np.arange(len(benchmarks_order))
width = 0.2

for i, strategy in enumerate(strategies):
    strategy_data = latency_df[latency_df['Strategy'] == strategy]
    latencies = []
    for benchmark in benchmarks_order:
        row = strategy_data[strategy_data['Benchmark'] == benchmark]
        if not row.empty:
            latencies.append(row['Avg Latency (ms)'].values[0])
        else:
            latencies.append(0)
    
    ax.bar(x + i * width, latencies, width, label=strategy)

ax.set_xlabel('Query Count', fontweight='bold', fontsize=12)
ax.set_ylabel('Average Latency (ms)', fontweight='bold', fontsize=12)
ax.set_title('Average Latency by Strategy and Query Count', fontweight='bold', fontsize=14)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(benchmarks_order)
ax.legend(loc='upper left', frameon=True, shadow=True)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### Figure 2: Precision & Recall Heatmap

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Precision heatmap
sns.heatmap(precision_pivot, annot=True, fmt='.4f', cmap='RdYlGn', 
            cbar_kws={'label': 'Precision'}, ax=axes[0], vmin=0, vmax=1)
axes[0].set_title('Average Precision', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Strategy', fontweight='bold')
axes[0].set_xlabel('Query Count', fontweight='bold')

# Recall heatmap
sns.heatmap(recall_pivot, annot=True, fmt='.4f', cmap='RdYlGn', 
            cbar_kws={'label': 'Recall'}, ax=axes[1], vmin=0, vmax=1)
axes[1].set_title('Average Recall', fontweight='bold', fontsize=12)
axes[1].set_ylabel('')
axes[1].set_xlabel('Query Count', fontweight='bold')

# F1 heatmap
sns.heatmap(f1_pivot, annot=True, fmt='.4f', cmap='RdYlGn', 
            cbar_kws={'label': 'F1 Score'}, ax=axes[2], vmin=0, vmax=1)
axes[2].set_title('Average F1 Score', fontweight='bold', fontsize=12)
axes[2].set_ylabel('')
axes[2].set_xlabel('Query Count', fontweight='bold')

plt.suptitle('Precision, Recall, and F1 Score Comparison', fontweight='bold', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### Figure 3: Latency Distribution (Box Plot)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Prepare data for box plot
latency_metrics_df = latency_df[['Strategy', 'Benchmark', 'Min Latency (ms)', 
                                   'Median Latency (ms)', 'Max Latency (ms)']]

# Create positions for each strategy-benchmark combination
positions = []
labels = []
data = []

pos = 0
for benchmark in benchmarks_order:
    for strategy in strategies:
        row = latency_df[(latency_df['Strategy'] == strategy) & 
                         (latency_df['Benchmark'] == benchmark)]
        if not row.empty:
            # Approximate distribution with min, median, max
            min_val = row['Min Latency (ms)'].values[0]
            med_val = row['Median Latency (ms)'].values[0]
            max_val = row['Max Latency (ms)'].values[0]
            
            data.append([min_val, med_val, max_val])
            positions.append(pos)
            labels.append(f"{strategy[:15]}\n{benchmark}")
        pos += 1
    pos += 1  # Add gap between benchmarks

bp = ax.boxplot(data, positions=positions, widths=0.6, patch_artist=True,
                showmeans=True, meanline=True)

# Color boxes by strategy
colors = plt.cm.Set3(np.linspace(0, 1, len(strategies)))
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor(colors[i % len(strategies)])
    patch.set_alpha(0.7)

ax.set_ylabel('Latency (ms)', fontweight='bold', fontsize=12)
ax.set_title('Latency Distribution by Strategy and Query Count', fontweight='bold', fontsize=14)
ax.set_xticks(positions[::len(strategies)])
ax.set_xticklabels(benchmarks_order, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Create legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colors[i], alpha=0.7, label=strategy) 
                   for i, strategy in enumerate(strategies)]
ax.legend(handles=legend_elements, loc='upper right', frameon=True, shadow=True)

plt.tight_layout()
plt.show()

## Export Tables for Presentation

Export all tables to CSV and Excel formats for use in presentations.

In [ ]:
# Create output directory
output_dir = Path('benchmark_tables')
output_dir.mkdir(exist_ok=True)

# Export individual pivot tables
tables = {
    'avg_latency': latency_pivot,
    'median_latency': median_latency_pivot,
    'p95_latency': p95_latency_pivot,
    'precision': precision_pivot,
    'recall': recall_pivot,
    'f1_score': f1_pivot
}

# Add IVFFlat tables if available
if not ivfflat_df.empty:
    tables['ivfflat_latency'] = ivfflat_latency_pivot
    tables['ivfflat_recall'] = ivfflat_recall_pivot

for name, table in tables.items():
    # CSV
    csv_path = output_dir / f"{name}_comparison.csv"
    table.to_csv(csv_path)
    print(f"✓ Exported: {csv_path}")

# Export comprehensive summary
summary_csv = output_dir / 'comprehensive_summary.csv'
summary_table.to_csv(summary_csv, index=False)
print(f"✓ Exported: {summary_csv}")

# Export IVFFlat raw data if available
if not ivfflat_df.empty:
    ivfflat_csv = output_dir / 'ivfflat_tuning_results.csv'
    ivfflat_df.to_csv(ivfflat_csv, index=False)
    print(f"✓ Exported: {ivfflat_csv}")

# Export to single Excel file with multiple sheets
excel_path = output_dir / 'benchmark_analysis_complete.xlsx'
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    # Strategy comparison sheets
    summary_table.to_excel(writer, sheet_name='Summary', index=False)
    latency_pivot.to_excel(writer, sheet_name='Avg Latency')
    median_latency_pivot.to_excel(writer, sheet_name='Median Latency')
    p95_latency_pivot.to_excel(writer, sheet_name='P95 Latency')
    precision_pivot.to_excel(writer, sheet_name='Precision')
    recall_pivot.to_excel(writer, sheet_name='Recall')
    f1_pivot.to_excel(writer, sheet_name='F1 Score')
    latency_df.to_excel(writer, sheet_name='Raw Latency Data', index=False)
    precision_recall_df.to_excel(writer, sheet_name='Raw Precision-Recall', index=False)
    
    # IVFFlat tuning sheets
    if not ivfflat_df.empty:
        ivfflat_latency_pivot.to_excel(writer, sheet_name='IVFFlat Latency')
        ivfflat_recall_pivot.to_excel(writer, sheet_name='IVFFlat Recall')
        ivfflat_df.to_excel(writer, sheet_name='IVFFlat Raw Data', index=False)

print(f"✓ Exported: {excel_path}")
print(f"\n✅ All tables exported to {output_dir}/ directory")

## Key Findings Summary

In [ ]:
print("\n" + "=" * 80)
print("KEY FINDINGS")
print("=" * 80)
print()

# Best latency strategy
best_latency = latency_df.loc[latency_df['Avg Latency (ms)'].idxmin()]
print(f"🏆 FASTEST STRATEGY:")
print(f"   {best_latency['Strategy']} ({best_latency['Benchmark']})")
print(f"   Average Latency: {best_latency['Avg Latency (ms)']:.2f} ms")
print()

# Best precision strategy
if not precision_recall_df.empty:
    best_precision = precision_recall_df.loc[precision_recall_df['Avg Precision'].idxmax()]
    print(f"🎯 HIGHEST PRECISION:")
    print(f"   {best_precision['Strategy']} ({best_precision['Benchmark']})")
    print(f"   Average Precision: {best_precision['Avg Precision']:.4f}")
    print()

    # Best recall strategy
    best_recall = precision_recall_df.loc[precision_recall_df['Avg Recall'].idxmax()]
    print(f"🔍 HIGHEST RECALL:")
    print(f"   {best_recall['Strategy']} ({best_recall['Benchmark']})")
    print(f"   Average Recall: {best_recall['Avg Recall']:.4f}")
    print()

    # Best F1 strategy
    best_f1 = precision_recall_df.loc[precision_recall_df['Avg F1'].idxmax()]
    print(f"⚖️  BEST F1 SCORE:")
    print(f"   {best_f1['Strategy']} ({best_f1['Benchmark']})")
    print(f"   Average F1: {best_f1['Avg F1']:.4f}")
    print()

# Latency trend across query counts
print(f"📈 LATENCY TRENDS:")
for strategy in latency_df['Strategy'].unique():
    strategy_data = latency_df[latency_df['Strategy'] == strategy].sort_values('Queries')
    if len(strategy_data) > 1:
        latencies = strategy_data['Avg Latency (ms)'].values
        trend = "increasing" if latencies[-1] > latencies[0] else "decreasing"
        print(f"   {strategy}: {trend} ({latencies[0]:.2f} → {latencies[-1]:.2f} ms)")

print()

# IVFFlat optimization summary
if not ivfflat_df.empty:
    print(f"⚙️  IVFFLAT OPTIMIZATION:")
    high_recall_configs = ivfflat_df[ivfflat_df['avg_recall'] >= 0.95]
    if not high_recall_configs.empty:
        optimal = high_recall_configs.loc[high_recall_configs['avg_query_time_ms'].idxmin()]
        print(f"   Optimal config: nlist={int(optimal['nlist'])}, nprobe={int(optimal['nprobe'])}")
        print(f"   Latency: {optimal['avg_query_time_ms']:.2f}ms")
        print(f"   Recall: {optimal['avg_recall']:.3f}")
        print(f"   Speedup vs exact search: {ivfflat_data['exact_search_avg_ms'] / optimal['avg_query_time_ms']:.1f}x")
    print()

print("=" * 80)